# Salifort Motors — Employee Retention Prediction

**Company:** Salifort Motors  
**Goal:** Predict which employees are likely to leave the company to reduce turnover and improve retention  
**Dataset:** HR_capstone_dataset.csv (14,999 rows, 10 columns)  
**Methodology:** PACE Framework (Plan → Analyze → Construct → Execute)  
**Tools:** Python (pandas, NumPy, matplotlib, seaborn, scikit-learn)  
**Model:** Random Forest Classifier  
**Final Accuracy:** 97%

---
## 1. Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

# Load dataset
df0 = pd.read_csv("HR_capstone_dataset.csv")
df0.head()

---
## 2. Initial Data Exploration

In [ ]:
# Display column names
df0.columns

In [ ]:
# Rename columns to fix typo
df0 = df0.rename(columns={'average_montly_hours': 'average_monthly_hours'})
df0.columns

In [ ]:
# Check for missing values
df0.isnull().sum()

In [ ]:
# Check for duplicates
df0.duplicated().sum()

In [ ]:
# Remove duplicates (3,008 duplicates = 20%)
df0 = df0.drop_duplicates()
print(f"Rows after deduplication: {len(df0)}")

---
## 3. Exploratory Data Analysis (EDA)

### Boxplot — Tenure Distribution and Outliers

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(x=df0['time_spend_company'])
plt.title('Distribution of Employee Tenure at Salifort Motors', fontsize=14, fontweight='bold')
plt.xlabel('Years at Company', fontsize=12)
plt.tight_layout()
plt.show()

# Calculate outliers
Q1 = df0['time_spend_company'].quantile(0.25)
Q3 = df0['time_spend_company'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
outliers = df0[(df0['time_spend_company'] < lower_bound) |
               (df0['time_spend_company'] > upper_bound)]
print(f"Number of rows with outliers: {len(outliers)}")

### Target Variable Distribution

In [ ]:
print("Count:")
print(df0['left'].value_counts())
print("\nPercentages:")
print(df0['left'].value_counts(normalize=True) * 100)

### Key Relationship Visualizations

In [ ]:
# 1. Satisfaction Level vs. Turnover
plt.figure(figsize=(10, 6))
sns.boxplot(x='left', y='satisfaction_level', data=df0)
plt.title('Satisfaction Level: Employees Who Left vs. Stayed', fontsize=14, fontweight='bold')
plt.xlabel('Employee Status (0=Stayed, 1=Left)', fontsize=12)
plt.ylabel('Satisfaction Level', fontsize=12)
plt.xticks([0, 1], ['Stayed', 'Left'])
plt.tight_layout()
plt.show()

In [ ]:
# 2. Average Monthly Hours vs. Turnover
plt.figure(figsize=(10, 6))
sns.boxplot(x='left', y='average_monthly_hours', data=df0)
plt.title('Average Monthly Hours: Employees Who Left vs. Stayed', fontsize=14, fontweight='bold')
plt.xlabel('Employee Status (0=Stayed, 1=Left)', fontsize=12)
plt.ylabel('Average Monthly Hours', fontsize=12)
plt.xticks([0, 1], ['Stayed', 'Left'])
plt.tight_layout()
plt.show()

In [ ]:
# 3. Number of Projects vs. Turnover
plt.figure(figsize=(10, 6))
sns.countplot(x='number_project', hue='left', data=df0)
plt.title('Number of Projects: Employees Who Left vs. Stayed', fontsize=14, fontweight='bold')
plt.xlabel('Number of Projects', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.legend(title='Left', labels=['Stayed', 'Left'])
plt.tight_layout()
plt.show()

In [ ]:
# 4. Satisfaction vs. Evaluation Scatterplot
plt.figure(figsize=(10, 6))
sns.scatterplot(x='satisfaction_level', y='last_evaluation', hue='left', data=df0, alpha=0.6)
plt.title('Satisfaction vs. Performance Evaluation', fontsize=14, fontweight='bold')
plt.xlabel('Satisfaction Level', fontsize=12)
plt.ylabel('Last Evaluation Score', fontsize=12)
plt.legend(title='Employee Status', labels=['Stayed', 'Left'])
plt.tight_layout()
plt.show()

In [ ]:
# 5. Correlation Heatmap
plt.figure(figsize=(10, 8))
correlation_matrix = df0.select_dtypes(include=[np.number]).corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Correlation Matrix of Numeric Variables', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. Random Forest Model

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Prepare the data
X = df0[['satisfaction_level', 'last_evaluation', 'number_project',
         'average_monthly_hours', 'time_spend_company', 'Work_accident',
         'promotion_last_5years']]
y = df0['left']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Build and train the model
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)

# Make predictions
y_pred = rf_model.predict(X_test)

In [ ]:
# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

In [ ]:
# Feature importance
feature_imp = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("Feature Importances:")
print(feature_imp)

In [ ]:
# Visualize confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - Random Forest')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

---
## 5. Key Findings & Business Recommendations

### Model Results
- **Accuracy:** 97%
- **Top predictors:** satisfaction_level (~42%), average_monthly_hours (~21%), number_project (~19%)
- **Precision:** 96% for predicting employees who leave
- **Recall:** 92% (catches 92% of employees who actually left)

### EDA Insights
1. **Satisfaction level is the strongest predictor** — employees who left show dramatically lower scores
2. **Overwork and underutilization both drive turnover** — 240+ hrs/month (burnout) and only 2 projects (disengagement)
3. **The 4–5 year tenure mark is critical** — turnover spikes when promotions don't materialize
4. **High performers are leaving** — cluster with high evaluation but low satisfaction
5. **Low salary and specific departments show elevated turnover**

### Recommendations
1. **Cap projects at 4–5 per employee** and flag anyone working 240+ hours/month
2. **Implement quarterly satisfaction pulse surveys** to identify at-risk employees early
3. **Create a 4-year retention program** with promotions or development opportunities
4. **Review compensation** for low-salary employees in high-turnover departments
5. **Deploy the model monthly** as an early-warning system for HR